# UX Frustration Recognition Experiment

In [1]:
%matplotlib widget
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import joblib

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import ADASYN, SMOTE

BASE_PATH = "../../experiment-data"
LABELS_SEPARATOR = ","
LABELS_FILENAME = "labels.csv"
FEATURES_SEPARATOR = ";"
FEATURES_DIRECTORY = f"{BASE_PATH}/extracted-features"
FEATURES_FILENAME = f"{FEATURES_DIRECTORY}/all_features.csv"
MODELS_DATE = "20250921"
MODELS_PATH = f"Results/{MODELS_DATE}"
RESULTS_SEPARATOR = ","
RESULTS_DIRECTORY = f"{BASE_PATH}/results"
BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]
DEBUG = True

### Import labels and dataset

In [2]:
labels = pd.read_csv(f"{BASE_PATH}/{LABELS_FILENAME}", sep=LABELS_SEPARATOR, header=0, index_col=0).dropna()
display(labels)

,binary-stress,affect3-class,affect4-class
subject/task,,,
01-AmusementClip,0,0,3
01-Baseline,0,0,0
01-EmoReset,0,0,0
01-FormL,1,2,1
01-FormM,1,2,1
...,...,...,...
21-Baseline,0,0,0
21-EmoReset,0,0,0
21-FormL,0,0,0


In [3]:
X = pd.read_csv(f"{FEATURES_FILENAME}", sep=FEATURES_SEPARATOR, header=0, index_col=0)
display(X)
for col in X.columns:
    X[col] = X[col].astype(np.float32)
display(X)


,meanHR,minHR,maxHR,sdHR,modeHR,nNN,meanNN,SDSD,CVNN,SDNN,...,min_scl,mean_scl,sd_scl,nSCR,aucSCR,meanAmpSCR,maxAmpSCR,meanRespSCR,sumAmpSCR,sumRespSCR
01-Baseline,67.938927,4.439306,99.096774,16.084483,94.657468,52.500000,1126.860119,1748.125453,1273.147961,1.129819,...,-0.760118,0.004440,0.990295,2.000000,-54.317275,0.610357,3.466212,0.830078,1.220715,1.245117
01-AmusementClip,68.058689,8.439560,83.027027,11.425651,74.587467,60.000000,993.598090,1046.284623,725.264597,0.729938,...,-1.047913,-0.000974,0.994654,2.000000,10.771605,0.189199,0.682391,3.294271,0.378399,6.588542
01-StressClip,69.354556,8.439560,87.771429,11.111625,79.331868,63.666667,937.397742,777.753528,553.581066,0.590551,...,-1.015111,0.001272,0.985412,1.666667,-11.291831,0.223363,0.461515,1.066406,0.372272,1.777344
01-EmoReset,67.862835,4.633484,78.769231,12.163410,74.135747,55.000000,1082.741477,1813.005039,1258.505426,1.162332,...,-1.020609,0.000963,0.996866,2.000000,-7.055041,0.166580,0.805823,0.777995,0.333160,1.555990
01-FormL,72.740026,10.072131,153.600000,12.343391,143.527869,66.800000,887.443862,642.268992,458.345263,0.516478,...,-1.359052,0.007178,0.948213,1.800000,-1.794321,1.011426,3.681678,13.962674,1.820567,25.132812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21-StressClip,79.547845,19.948052,87.771429,7.850818,67.823377,72.988384,780.092593,383.750433,272.192135,0.348923,...,-1.233190,0.001964,0.950427,0.540655,-2.007692,3.065072,3.065072,7.656250,1.657146,4.139388
21-EmoReset,82.437562,13.963636,90.352941,5.186123,76.389305,80.655325,740.546875,319.267953,226.755158,0.306200,...,-1.474851,0.004269,0.937666,3.548834,-17.690004,0.742909,2.196171,2.228338,2.636461,7.908003
21-FormL,79.716174,11.861004,170.666667,8.736983,158.805663,77.612075,771.702776,371.389631,262.132710,0.339681,...,-1.284004,0.001639,0.987437,2.727581,-9.724188,0.365968,1.106248,9.481534,0.998207,25.861651
21-FormM,82.096186,28.444444,93.090909,4.388562,64.646465,81.368452,734.986034,102.934253,85.176432,0.115889,...,-1.582827,0.005257,0.804585,5.757917,-79.122038,1.287057,3.148032,1.746962,7.410764,9.529447


,meanHR,minHR,maxHR,sdHR,modeHR,nNN,meanNN,SDSD,CVNN,SDNN,...,min_scl,mean_scl,sd_scl,nSCR,aucSCR,meanAmpSCR,maxAmpSCR,meanRespSCR,sumAmpSCR,sumRespSCR
01-Baseline,67.938927,4.439306,99.096771,16.084482,94.657471,52.500000,1126.860107,1748.125488,1273.147949,1.129819,...,-0.760118,0.004440,0.990295,2.000000,-54.317276,0.610357,3.466212,0.830078,1.220715,1.245117
01-AmusementClip,68.058685,8.439561,83.027023,11.425651,74.587463,60.000000,993.598083,1046.284668,725.264587,0.729938,...,-1.047913,-0.000974,0.994654,2.000000,10.771605,0.189199,0.682391,3.294271,0.378399,6.588542
01-StressClip,69.354553,8.439561,87.771431,11.111626,79.331871,63.666668,937.397766,777.753540,553.581055,0.590551,...,-1.015111,0.001272,0.985412,1.666667,-11.291831,0.223363,0.461515,1.066406,0.372272,1.777344
01-EmoReset,67.862831,4.633484,78.769234,12.163410,74.135750,55.000000,1082.741455,1813.005005,1258.505371,1.162332,...,-1.020609,0.000963,0.996866,2.000000,-7.055041,0.166580,0.805823,0.777995,0.333160,1.555990
01-FormL,72.740028,10.072131,153.600006,12.343390,143.527863,66.800003,887.443848,642.268982,458.345276,0.516478,...,-1.359052,0.007178,0.948213,1.800000,-1.794321,1.011426,3.681678,13.962673,1.820567,25.132812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21-StressClip,79.547844,19.948051,87.771431,7.850818,67.823380,72.988388,780.092590,383.750427,272.192139,0.348923,...,-1.233190,0.001964,0.950427,0.540655,-2.007692,3.065072,3.065072,7.656250,1.657146,4.139388
21-EmoReset,82.437561,13.963636,90.352943,5.186123,76.389305,80.655327,740.546875,319.267944,226.755157,0.306200,...,-1.474851,0.004269,0.937666,3.548834,-17.690004,0.742909,2.196171,2.228338,2.636461,7.908002
21-FormL,79.716171,11.861004,170.666672,8.736983,158.805664,77.612076,771.702759,371.389618,262.132721,0.339681,...,-1.284004,0.001639,0.987437,2.727581,-9.724188,0.365968,1.106248,9.481534,0.998207,25.861652
21-FormM,82.096184,28.444445,93.090912,4.388562,64.646461,81.368454,734.986023,102.934250,85.176430,0.115889,...,-1.582827,0.005257,0.804585,5.757916,-79.122040,1.287057,3.148031,1.746962,7.410764,9.529447


In [4]:
# Selecting rows that actually have entries in "labels" file
idx = list(X.merge(labels, left_index=True, right_index=True).index)
labels = labels.loc[idx]
x = X.loc[idx]
# Debriefing task has no questionnaire data, it could be used as unseen data and labelled as expected "relax" without ground truth
print(f"Selected {len(x)} entries from X. Not considering {len(X) - len(x)} entries.")

Selected 126 entries from X. Not considering 21 entries.


### Helper functions

In [5]:
def make_results_filename(classification: str, feature_selection: str | None) -> str:
    now = datetime.now()
    timestamp = now.strftime("%Y%m%d-%H%M%S")
    feature_selection = feature_selection if feature_selection is not None else "None"
    return f"{RESULTS_DIRECTORY}/{timestamp}_{classification}_{feature_selection}f"


def show_label_stats(labels: list[str], y: pd.Series):
    display(
        pd.DataFrame(
            {
                "labels": labels,
                "counts": y.value_counts().to_list(),
                "percentage": (y.value_counts(normalize=True) * 100).to_list(),
            }
        )
    )

def show_results(classification: str, feature_selector: str, results: pd.DataFrame, save=False):
    display(results)
    if save:
        res_file = make_results_filename(classification, feature_selector) + ".csv"
        print(f"Saving results to {res_file}")
        results.to_csv(res_file, sep=RESULTS_SEPARATOR)

def show_confusion_matrices(classification: str, feature_selector: str, conf_matrices: dict[str, np.ndarray], labels: list[int], save=False):
    cm_file = make_results_filename(classification, feature_selector) + ".CM.png"
    ncms = len(conf_matrices)
    pairs = [(name, cm) for name, cm in conf_matrices.items()]
    nrows = (ncms // 2) + (ncms % 2)
    fig, axes = plt.subplots(nrows, 2, figsize=(8, nrows * 4))
    for (name, cm), ax in zip(pairs, axes.ravel()):
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)
        ax.set_title(name)
    fig.suptitle("Confusion Matrices")
    plt.tight_layout()
    if save:
        print(f"Saving confusion matrix to {cm_file}")
        plt.savefig(cm_file, dpi=300, format="png")
    plt.show()



### Parameters

In [ ]:
params = {
    "show_confusion_matrices": True,
    "save_confusion_matrices": True,
    "balance_dataset": True,
    "save_results": False,
    "verbose": False,
}

### Classification

In [ ]:
feat_sel = "RFE"
models_paths = {
    "b": {
        "LogisticRegression": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_LogisticRegression_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_MLPClassifier_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_SVC_{feat_sel}f.joblib",
    },
    "t": {
        "LogisticRegression": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_LogisticRegression_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_MLPClassifier_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_SVC_{feat_sel}f.joblib",
    },
    "q": {
        "LogisticRegression": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_LogisticRegression_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_MLPClassifier_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_SVC_{feat_sel}f.joblib",
    },
}
models: dict[str, dict[str, Pipeline]] = {}
for class_type, paths in models_paths.items():
    models[class_type] = {}
    for model_type, path in paths.items():
        models[class_type][model_type] = joblib.load(path)

for class_type, models in models.items():
    if class_type == "b":
        params["classification"] = "binary"
        params["classes"] = BIN_LABELS
        y = labels["binary-stress"]
    elif class_type == "t":
        params["classification"] = "ternary"
        params["classes"] = TER_LABELS
        y = labels["affect3-class"]
    elif class_type == "q":
        params["classification"] = "quaternary"
        params["classes"] = QAD_LABELS
        y = labels["affect4-class"]

    print(f"\n\n============ {params['classification']} ============\n\n")
    show_label_stats(params["classes"], y)

    df_res = pd.DataFrame(
        {
            "classifier": [],
            "bal-accuracy": [],
            "weighted-f1": [],
            "macro-avg-prec": [],
            "macro-avg-rec": [],
            "macro-avg-f1": [],
            "stress-prec": [],
            "stress-rec": [],
            "stress-f1": [],
        }
    )
    conf_matrices: dict[str, np.ndarray] = {}

    for model_name, model in models.items():
        y_pred = model.predict(x)
        cm = confusion_matrix(y, y_pred)
        report = classification_report(y, y_pred, target_names=params["classes"], output_dict=True)
        conf_matrices[model_name] = cm

        s_report = report["Stress"]
        m_report = report["macro avg"]
        new_row = {
            "classifier": model_name,
            "bal-accuracy": balanced_accuracy_score(y, y_pred),
            "weighted-f1": f1_score(y, y_pred, average="weighted"),
            "macro-avg-prec": m_report["precision"],
            "macro-avg-rec": m_report["recall"],
            "macro-avg-f1": m_report["f1-score"],
            "stress-prec": s_report["precision"],
            "stress-rec": s_report["recall"],
            "stress-f1": s_report["f1-score"],
        }
        df_res.loc[len(df_res)] = new_row

    show_results(params["classification"], feat_sel, df_res, save=params["save_results"])
    if params["show_confusion_matrices"]:
        show_confusion_matrices(params["classification"], feat_sel, conf_matrices, labels=params["classes"], save=params["save_confusion_matrices"])